# シミュレーション環境の構築

## 前提

- macOS か Windows を使用

## GitとGit LFSのインストール

```sh
# 🍎 macOS
# Homebrewをインストール
/bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"
echo 'eval "$(/opt/homebrew/bin/brew shellenv)"' >> ~/.zprofile
eval "$(/opt/homebrew/bin/brew shellenv)"
brew --version
brew install git git-lfs
git lfs install
```

```sh
# 🪟 Windows
# Git for Windowsをインストール https://git-scm.com/install/windows
git lfs install
```

## UVのインストール

```sh
# 🍎 macOS
curl -LsSf https://astral.sh/uv/install.sh | sh

# 🪟 Windows
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"

# インストール後ターミナルを開き直し、検証
uv --version
# v0.11.18 古い場合は uv self update を実行
# もしくは curl -LsSf https://astral.sh/uv/install.sh | sh を再実行
```

## 仮想環境を構築

```sh
# 仮想環境を作成
uv venv .reachy_mini_env --python 3.12

# macOS
source .reachy_mini_env/bin/activate

# Windows
# Win + R を押し、powershellと入力。Windows Powershellを右クリックし、管理者として実行
powershell Set-ExecutionPolicy RemoteSigned
# Powershellを閉じ、通常のターミナルを開く
reachy_mini_env\Scripts\activate
```

## Reachy Mini SDK をインストール

https://github.com/pollen-robotics/reachy_mini

In [ ]:
import os

if not os.path.exists("reachy_mini"):
    !git clone https://github.com/pollen-robotics/reachy_mini

!cd reachy_mini && git describe --tags # v1.9.0-91-g234a978e4

In [ ]:
!uv pip install "reachy-mini[mujoco]"

In [ ]:
!uv pip show reachy-mini # 1.11.0.dev0

## シミュレーション環境の実行

仮想環境を有効化

```sh
# macOS
source .reachy_mini_env/bin/activate

# Windows
reachy_mini_env\Scripts\activate
```

シミュレータを起動する。`Ctrl + c`で停止が可能。

```sh
# macOS
mjpython -m reachy_mini.daemon.app.main --sim

# Windows
reachy-mini-daemon --sim
```

シーンを適用したシミュレータを起動する。

```sh
# macOS
mjpython -m reachy_mini.daemon.app.main --sim --scene minimal

# Windows
reachy-mini-daemon --sim --scene minimal
```

シミュレータが起動した状態でロボットを動かす。

In [ ]:
from reachy_mini import ReachyMini
from reachy_mini.utils import create_head_pose

# localhost で実行中のシミュレーションに接続
# with は使わず、このノートブック全体でこの mini を使い回す（同一プロセス内で
# ReachyMini() の open/close を繰り返すとカメラが壊れるバグの回避策）。
# 使い終わったら一番下のセルで mini.close() すること。
mini = ReachyMini(media_backend="default")
print("シミュレーションに接続しました！")

# 見上げて首を傾ける
print("頭部を移動中...")
mini.goto_target(
    head=create_head_pose(z=20, roll=10, mm=True, degrees=True),
    duration=1.0
)

# アンテナを動かす
print("アンテナを動かしています...")
mini.goto_target(antennas=[0.6, -0.6], duration=0.3)
mini.goto_target(antennas=[-0.6, 0.6], duration=0.3)

# 初期位置に戻す
mini.goto_target(
    head=create_head_pose(),
    antennas=[0, 0],
    duration=1.0
)
print("初期位置に戻しました！")

## GStreamerの確認

GStreamerは、マイク・スピーカー・カメラと連携するためのライブラリ。

新しいターミナルを開き、仮想環境を有効化する。

```sh
# macOS
source .reachy_mini_env/bin/activate

# Windows
reachy_mini_env\Scripts\activate
```

GStreamerのバージョンを確認する。

```sh
gst-inspect-1.0 --version # GStreamer 1.28.3
```

## GStreamerの初期化

In [ ]:
import gi
gi.require_version("Gst", "1.0")
from gi.repository import Gst

Gst.init(None)
print("GStreamer:", Gst.version_string(), "\n")

プラグインが正しくインストールされているかを確認する。

In [ ]:
REQUIRED = {
    "webrtcsink":        "WebRTC配信（daemon → 遠隔クライアント）",
    "webrtcbin":         "WebRTC下位実装",
    "webrtcdsp":         "エコーキャンセル(AEC)",
    "webrtcechoprobe":   "AEC参照信号プローブ",
    "unixfdsink":        "IPC送信（LOCALバックエンド, mac/Linux）",
    "unixfdsrc":         "IPC受信（LOCALバックエンド, mac/Linux）",
    "equalizer-10bands": "スピーカーEQ",
    "audiodynamic":      "EQ後のリミッタ",
    "appsink":           "GStreamer → numpy",
    "appsrc":            "numpy → GStreamer",
    "jpegdec":           "USBカメラのMJPEGデコード",
}

for name, desc in REQUIRED.items():
    f = Gst.ElementFactory.find(name)
    mark = "✅" if f else "❌"
    plugin = f" (plugin: {f.get_plugin_name()})" if f else ""
    print(f"{mark} {name:<18} {desc}{plugin}")

## デバイスの確認

利用可能な入力音声デバイス一覧（audio source）を表示する。

In [ ]:
from reachy_mini.media.device_detection import gst_monitor_devices

for d in gst_monitor_devices("Audio/Source"):
    print(f"[{d.index}] {d.display_name}")
    print(f"class={d.device_class}  props={dict(list(d.properties.items())[:4])}")

利用可能な出力音声デバイス一覧（audio sink）を表示する。

In [ ]:
for d in gst_monitor_devices("Audio/Sink"):
    print(f"[{d.index}] {d.display_name}")
    print(f"class={d.device_class}  props={dict(list(d.properties.items())[:4])}")

利用可能なカメラデバイス一覧（video source）を表示する。

In [ ]:
for d in gst_monitor_devices("Video/Source"):
    print(f"[{d.index}] {d.display_name}")
    print(f"class={d.device_class}  props={dict(list(d.properties.items())[:4])}")

ロボットが使用しているデバイスを表示する。

In [ ]:
from reachy_mini.media.device_detection import get_audio_device, get_video_device

mic_id   = get_audio_device("Source")
spk_id   = get_audio_device("Sink")
cam_path, cam_specs = get_video_device()

print(f"Microphone : {mic_id or '(not found / sim モードでは未使用)'}")
print(f"Speaker    : {spk_id or '(not found / sim モードでは未使用)'}")
print(f"Camera     : {cam_path or '(not found — sim モードは MuJoCo UDP 経由)'}")

In [ ]:
# 現在のオーディオ音量を確認

from reachy_mini.daemon.app.routers.volume_control import get_volume_control

vc = get_volume_control()

output_vol = vc.get_output_volume()
input_vol  = vc.get_input_volume()

print(f"プラットフォーム : {vc.platform_name}")
print(f"スピーカー  ({vc.output_device.name}): {output_vol}%")
print(f"マイク      ({vc.input_device.name}) : {input_vol}%")


In [ ]:
# スピーカーとマイクの音量を100%に設定

from reachy_mini.daemon.app.routers.volume_control import get_volume_control

vc = get_volume_control()

ok_out = vc.set_output_volume(100)
ok_in  = vc.set_input_volume(100)

print(f"スピーカー  : {'✓ 100%' if ok_out else '✗ 設定失敗'}")
print(f"マイク      : {'✓ 100%' if ok_in  else '✗ 設定失敗'}")

# 確認
print(f"\n設定後 — スピーカー: {vc.get_output_volume()}%  マイク: {vc.get_input_volume()}%")


In [ ]:
if cam_specs:
    print(f"カメラの仕様: {cam_specs}")
else:
    print("カメラの仕様は取得できませんでした。")

In [ ]:
# カメラの解像度とフレームレートの一覧

from reachy_mini.media.camera_constants import (
    MujocoCameraSpecs,
    ReachyMiniLiteCamSpecs,
    ReachyMiniWirelessCamSpecs,
    ArducamSpecs,
)

for specs_cls in [MujocoCameraSpecs, ReachyMiniLiteCamSpecs, ReachyMiniWirelessCamSpecs, ArducamSpecs]:
    specs = specs_cls()
    print(f"\n【{specs.name}】 デフォルト: {specs.default_resolution.name}")
    for r in specs.available_resolutions:
        w, h, fps, _ = r.value
        marker = " ← default" if r == specs.default_resolution else ""
        print(f"  {w:4d}x{h:<4d} @{fps:2d}fps  ({r.name}){marker}")


## 音声の録音

音声を録音する。

In [ ]:
from scipy.signal import resample
import time
import numpy as np

# 録音パイプラインを開始
mini.media.start_recording()
record_seconds = 5

# マイクのサンプリングレート（Hz）を取得
in_sr  = mini.media.get_input_audio_samplerate()

# マイクのチャンネル数を取得
in_ch  = mini.media.get_input_channels()

# 録音するサンプル数を計算
target_samples = int(record_seconds * in_sr)

# スピーカーのサンプリングレート（Hz）を取得
out_sr = mini.media.get_output_audio_samplerate()

print(f"音声を{record_seconds}秒間録音中... ({in_sr} Hz, {in_ch}ch)")

audio_samples = []
collected = 0

while collected < target_samples:
    samples = mini.media.get_audio_sample()
    if samples is not None:
        audio_samples.append(samples)
        collected += len(samples)
        print(f"\rサンプル取得中: {collected/in_sr:.1f}s", end="")
    else:
        time.sleep(0.01)
print()
print(f"{len(audio_samples)} 件のサンプルを取得しました。")

# 録音パイプラインを停止
mini.media.stop_recording()

audio_data = np.concatenate(audio_samples, axis=0)[:target_samples]
print(f"Recorded audio: {len(audio_data)} samples at {in_sr} Hz, {in_ch}ch")

In [ ]:
# 録音データの検証

x = audio_data.astype(np.float64)

def to_db(v):
    return 20 * np.log10(v) if v > 0 else float("-inf")

peak = float(np.abs(x).max())
rms  = float(np.sqrt(np.mean(x**2)))

print(f"shape       : {audio_data.shape}   dtype: {audio_data.dtype}")
print(f"長さ         : {len(x)/in_sr:.2f} 秒")
print(f"ピーク       : {peak:.6f}  ({to_db(peak):+.1f} dBFS)")
print(f"RMS          : {rms:.6f}  ({to_db(rms):+.1f} dBFS)")
print(f"非ゼロ比率   : {np.count_nonzero(x) / x.size:.1%}")
print(f"NaN / Inf    : {np.isnan(x).any()} / {np.isinf(x).any()}")

for ch in range(x.shape[1]):
    c = x[:, ch]
    print(f"  ch{ch}: peak={np.abs(c).max():.6f}  rms={np.sqrt(np.mean(c**2)):.6f}")

print()
if np.isnan(x).any() or np.isinf(x).any():
    print("❌ NaN / Inf が含まれています")
elif peak == 0.0:
    print("❌ 完全な無音（全サンプルが厳密に 0）→ マイクからデータが届いていません")
elif peak < 1e-3:
    print("⚠️  ほぼ無音（-60 dBFS 未満）→ ゲイン不足か別デバイスの可能性")
elif peak >= 1.0:
    print("⚠️  クリップしています（音が割れます）")
else:
    print("✅ 音声が録れています")


In [ ]:
from IPython.display import Audio

mono = audio_data.mean(axis=1)
display(Audio(data=mono, rate=out_sr))

## 音声の再生

音声をスピーカーで再生する。

In [ ]:
out_sr = mini.media.get_output_audio_samplerate()

data = audio_data
# 入出力でレートが違う場合だけリサンプリング（現状はどちらも16kHzなので通らない）
if in_sr != out_sr:
    data = resample(data, int(len(data) * out_sr / in_sr))

# push_audio_sample は float32 を要求する（resample は float64 を返すため必須）
data = np.ascontiguousarray(data, dtype=np.float32)

# 再生パイプラインを開始
mini.media.start_playing()
print(f"再生中... ({len(data)/out_sr:.1f}s, {out_sr} Hz)")

# チャンクに分けて送出
chunk_size = 1024
for i in range(0, len(data), chunk_size):
    mini.media.push_audio_sample(data[i : i + chunk_size])

# 再生完了を待つ（push は非同期なので待たないと途中で切れる）
time.sleep(len(data) / out_sr + 0.3)

# 再生パイプラインを停止
mini.media.stop_playing()
print("再生完了")

音声をWAVに保存する。

In [ ]:
import soundfile as sf
from reachy_mini.media.gstreamer_utils import audio_duration_seconds

path = "recorded.wav"
sf.write(path, audio_data, in_sr)   # in_sr で書けばファイル側にレート情報が入る


WAVを再生する。

In [ ]:
import os

mini.media.play_sound(os.path.abspath(path))   # 絶対パスで渡す
time.sleep(audio_duration_seconds(path) + 0.3)

## カメラで撮影する

In [ ]:
!uv pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

frame = mini.media.get_frame()
plt.imshow(frame[:, :, ::-1])  # BGR -> RGB に変換して表示
plt.axis("off")
plt.title(f"frame {frame.shape}, {frame.dtype}")
plt.show()

## OpenAI API キーのセットアップ

`.env` ファイルを作成し、以下を記載する。

```sh
OPENAI_API_KEY=sk-proj-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
```

APIキーを読み込む。

In [ ]:
!uv pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

assert OPENAI_API_KEY, "OPENAI_API_KEY が設定されていません。`.env` を作成してください。"

## 音声からテキストを抽出

`gpt-4o-transcribe`を使用し、音声からテキストを抽出する。

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

with open(path, "rb") as audio_file:
    transcript = client.audio.transcriptions.create(
        model="gpt-4o-mini-transcribe-2025-12-15",
        file=audio_file,
        language="ja",
    )

print(transcript.text)


## 抽出したテキストから回答を生成

In [ ]:
REACHY_MINI_SYSTEM_PROMPT = """\
あなたは「Reachy Mini」という卓上サイズの小型ロボットです。
丸い頭とアンテナ、かわいい見た目を持ち、話しながら頭やアンテナをちょこちょこ動かします。
好奇心旺盛で人懐っこく、親しみやすい相棒として振る舞ってください。

出力は音声合成でそのまま読み上げられます。次のルールを守ってください。
- 1〜2文程度の短い口語体で、テンポよく答える
- 絵文字・記号・箇条書き・かっこ書きの補足は使わない（音声にならないため）
- 難しい専門用語は避け、話しかけるような自然な話し言葉にする
- 日本語で答える
"""

response = client.responses.create(
    model="gpt-5-nano",
    instructions=REACHY_MINI_SYSTEM_PROMPT,
    input=transcript.text,
)

print(response.output_text)


## 回答から音声を生成

In [ ]:
speech_path = "response.wav"

# playbin（mini.media.play_sound）がそのまま再生できるようWAVで受け取る
with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="alloy",
    input=response.output_text,
    response_format="wav",
) as tts_response:
    tts_response.stream_to_file(speech_path)

mini.media.play_sound(os.path.abspath(speech_path))
time.sleep(audio_duration_seconds(speech_path) + 0.3)


## 発話区間の抽出（VAD）

Silero VAD（`torch.hub`経由）を使い、録音した音声（`audio_data`）から人が話している区間だけを検出する。
torchのみで動くローカル処理（クラウドAPI不要）なので、macOS・Windowsどちらでも同様に動作する。
初回実行時はモデルを一度ダウンロードしてキャッシュし、以降はオフラインで動く。

In [ ]:
!uv pip install torch

In [ ]:
import torch

torch.set_num_threads(1)

# Silero VAD（初回はモデルをダウンロードしてキャッシュ、以降はローカルのみで動作）
vad_model, vad_utils = torch.hub.load(
    "snakers4/silero-vad",
    "silero_vad",
    trust_repo=True,
    skip_validation=True,
)
get_speech_timestamps, _, _, _, _ = vad_utils

# Silero VAD は 8000Hz/16000Hz・モノラルのみ対応
if in_sr not in (8000, 16000):
    raise ValueError(f"Silero VAD は8000/16000Hzのみ対応（in_sr={in_sr}）")
mono = audio_data.mean(axis=1).astype(np.float32)

speech_timestamps = get_speech_timestamps(
    torch.from_numpy(mono),
    vad_model,
    sampling_rate=in_sr,
    threshold=0.5,
)

print(f"検出された発話区間: {len(speech_timestamps)} 件")
for i, seg in enumerate(speech_timestamps, start=1):
    start_s = seg["start"] / in_sr
    end_s = seg["end"] / in_sr
    print(f"  {i}. {start_s:.2f}s 〜 {end_s:.2f}s （{end_s - start_s:.2f}秒）")

In [ ]:
speech_segments = [mono[seg["start"]:seg["end"]] for seg in speech_timestamps]
speech_only = np.concatenate(speech_segments) if speech_segments else np.array([], dtype=np.float32)

print(f"発話区間だけの音声: {len(speech_only)/in_sr:.2f}秒（元の録音: {len(mono)/in_sr:.2f}秒）")
display(Audio(data=speech_only, rate=in_sr))

## カメラ画像を使ったFunction Calling

`gpt-5-nano`（Responses API）にfunction tool `capture_camera_image` を渡す。
「これ何？」「今何が見える？」のように目の前の状況について聞かれたときだけ、モデルが自分でこのツールを呼び出す。
ツールが呼ばれたら reachy_mini SDK（`mini.media.get_frame()`）でカメラ画像を1枚取得し、`function_call_output` の画像コンテンツとしてモデルに返す。

In [ ]:
import base64
from io import BytesIO

from PIL import Image

CAMERA_TOOL = {
    "type": "function",
    "name": "capture_camera_image",
    "description": (
        "Reachy Miniのカメラで現在の映像を1枚撮影して確認する。"
        "「これ何？」「今何が見える？」など、目の前の物や状況について"
        "答える必要があるときに呼び出す。"
    ),
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False,
    },
    "strict": True,
}


def capture_camera_image_data_url() -> str:
    """reachy_mini SDKでカメラ画像を1枚取得し、data URL（JPEG, base64）にして返す。"""
    frame = mini.media.get_frame()  # BGR, shape (H, W, 3)
    if frame is None:
        raise RuntimeError("カメラ画像を取得できませんでした。")

    rgb = frame[:, :, ::-1]
    buf = BytesIO()
    Image.fromarray(rgb).save(buf, format="JPEG")
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}"

In [ ]:
user_input = "今カメラに何が見えるか教えて"

response = client.responses.create(
    model="gpt-5-nano",
    instructions=REACHY_MINI_SYSTEM_PROMPT,
    tools=[CAMERA_TOOL],
    input=user_input,
)

image_calls = [
    item
    for item in response.output
    if item.type == "function_call" and item.name == "capture_camera_image"
]

if image_calls:
    print("カメラ撮影の呼び出しが検出されました。画像を取得して回答に反映します...")
    call = image_calls[0]
    image_data_url = capture_camera_image_data_url()

    response = client.responses.create(
        model="gpt-5-nano",
        previous_response_id=response.id,
        input=[
            {
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": [
                    {"type": "input_image", "image_url": image_data_url, "detail": "auto"},
                ],
            }
        ],
    )

print(response.output_text)

## ロボットの動きを使ったFunction Calling

`gpt-5-nano` にfunction tool `play_robot_move` を渡す。感情を込めたい、ダンスを頼まれた、など体の動きで
表現した方が良い場面で、モデルが必要に応じて自分でこのツールを呼び出す。
呼ばれたら reachy_mini SDK の `RecordedMoves`（ダンス・感情ライブラリ）から該当の録画済みモーションを取得し、
`mini.play_move()` で再生する。

In [ ]:
from reachy_mini.motion.recorded_move import RecordedMoves

# 初回はHuggingFaceからダウンロードしてキャッシュ（daemon起動時にプリロード済みのことが多い）
dances = RecordedMoves("pollen-robotics/reachy-mini-dances-library")
emotions = RecordedMoves("pollen-robotics/reachy-mini-emotions-library")

# 実際にHFからロードできた動きだけを対象にする（名前のハードコードはしない）
MOVE_REGISTRY = {name: dances for name in dances.list_moves()}
MOVE_REGISTRY.update({name: emotions for name in emotions.list_moves()})

PLAY_MOVE_TOOL = {
    "type": "function",
    "name": "play_robot_move",
    "description": (
        "Reachy Miniの体でダンスや感情表現の録画済みモーションを再生する。"
        "感情を込めたい、またはユーザーにダンスを頼まれたときなど、"
        "体の動きで表現した方が良い場面でだけ呼び出す。"
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "move_name": {
                "type": "string",
                "enum": sorted(MOVE_REGISTRY),
                "description": "再生する動きの名前",
            },
        },
        "required": ["move_name"],
        "additionalProperties": False,
    },
    "strict": True,
}


async def play_robot_move(move_name: str) -> str:
    """move_nameに対応する録画済みモーションをReachy Miniで再生する。

    Jupyterはセルの実行自体が非同期イベントループ上で動くため、同期ラッパーの
    `mini.play_move()`（内部でAsyncToSyncを使う）は
    "AsyncToSync in the same thread as an async event loop" エラーになる。
    ここでは非同期版の `async_play_move()` を直接awaitする。
    """
    registry = MOVE_REGISTRY.get(move_name)
    if registry is None:
        return f"不明な動き: {move_name}"

    move = registry.get(move_name)
    await mini.async_play_move(move, initial_goto_duration=1.0)
    return f"{move_name} を再生しました。"

In [ ]:
import json


async def run_tool_call(call) -> str | list[dict]:
    """function_callアイテムを実行し、function_call_outputのoutputを返す。"""
    args = json.loads(call.arguments) if call.arguments else {}

    if call.name == "capture_camera_image":
        image_data_url = capture_camera_image_data_url()
        return [{"type": "input_image", "image_url": image_data_url, "detail": "auto"}]

    if call.name == "play_robot_move":
        return await play_robot_move(args["move_name"])

    raise ValueError(f"未対応のツール: {call.name}")


user_input = "元気な感じで踊ってみて"

response = client.responses.create(
    model="gpt-5-nano",
    instructions=REACHY_MINI_SYSTEM_PROMPT,
    tools=[CAMERA_TOOL, PLAY_MOVE_TOOL],
    input=user_input,
)

# モデルがツールを呼び出す限り実行し、結果を返し続ける
function_calls = [item for item in response.output if item.type == "function_call"]

while function_calls:
    outputs = []
    for call in function_calls:
        outputs.append({
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": await run_tool_call(call),
        })

    response = client.responses.create(
        model="gpt-5-nano",
        previous_response_id=response.id,
        input=outputs,
    )
    function_calls = [item for item in response.output if item.type == "function_call"]

print(response.output_text)

## 感情表現豊かなシステムプロンプト

`REACHY_MINI_SYSTEM_PROMPT`はそのままに、`play_robot_move`をほぼ毎回使うよう強く指示した
別の定数 `REACHY_MINI_EXPRESSIVE_SYSTEM_PROMPT` を用意する。会話デモではこちらを使う。

In [ ]:
REACHY_MINI_EXPRESSIVE_SYSTEM_PROMPT = """\
あなたは「Reachy Mini」という卓上サイズの小型ロボットです。
丸い頭とアンテナ、かわいい見た目を持ち、話しながら頭やアンテナをちょこちょこ動かします。
好奇心旺盛で人懐っこく、親しみやすい相棒として振る舞ってください。

出力は音声合成でそのまま読み上げられます。次のルールを守ってください。
- 1〜2文程度の短い口語体で、テンポよく答える
- 絵文字・記号・箇条書き・かっこ書きの補足は使わない（音声にならないため）
- 難しい専門用語は避け、話しかけるような自然な話し言葉にする
- 日本語で答える

あなたは体を使った感情表現がとても豊かなロボットです。次のルールで積極的に play_robot_move を使ってください。
- 返答するときは、ほぼ毎回 play_robot_move を1回呼び出し、返答の内容や感情に合った動きを再生してから答える
  （例: 相槌や同意には yes1 や simple_nod、驚いたときは amazed1 や surprised1、
  嬉しい・楽しいときは enthusiastic1 やダンス系の動き、悲しい・残念なときは sad1、
  ダンスを頼まれたときはダンス系の動き）
- 「特別な感情がない」と思える受け答えでも、ちょっとした相槌の動き（yes1やsimple_nodなど）を選んで使う
- 同じ動きばかりにならないよう、文脈に応じて毎回違う動きを選ぶ
- capture_camera_image は目の前の状況を実際に確認する必要があるときだけ呼ぶ
"""

## しゃべっている間の頭の動き（Head Wobbling）を有効化

`mini.enable_wobbling()` を一度呼んでおくと、以降 `mini.media.play_sound()` /
`mini.media.push_audio_sample()` で再生する音声がリアルタイム解析され、音量・発話区間に応じた
頭の揺れ（sway/roll）が自動で合成される。つまりTTSの音声を再生するたびに「喋っている感」の
動きが自動で付くようになる。会話デモのTTS再生（`mini.media.play_sound(...)`）にもそのままかかる。

In [ ]:
mini.enable_wobbling()
print("Head wobbling を有効化しました。")

## 会話デモ（VAD → STT → Response API → TTS）

録音→VADで発話区間抽出→STT→応答生成→TTS再生、を`for`ループで複数ターン繰り返すシンプルなデモ。
Response APIは`previous_response_id`で前のターンの応答につなげているため、過去の会話が積み上がった状態で応答が生成される
（会話全体を毎回送り直す必要はない）。`tools=[CAMERA_TOOL, PLAY_MOVE_TOOL]`も渡しているので、
必要に応じてモデルが自分でカメラ撮影やダンス・感情表現を呼び出す。
システムプロンプトは`REACHY_MINI_SYSTEM_PROMPT`ではなく`REACHY_MINI_EXPRESSIVE_SYSTEM_PROMPT`を使い、
ほぼ毎回の返答で`play_robot_move`を使うよう促している。

上のセルで定義済みの `mini` / `client` / `vad_model` / `get_speech_timestamps` /
`REACHY_MINI_EXPRESSIVE_SYSTEM_PROMPT` / `CAMERA_TOOL` / `PLAY_MOVE_TOOL` / `run_tool_call` をそのまま使う。

In [ ]:
N_TURNS = 3
record_seconds = 4

previous_response_id = None

for turn in range(1, N_TURNS + 1):
    print(f"\n=== ターン {turn}/{N_TURNS} ===")

    # 1. 録音
    mini.media.start_recording()
    turn_in_sr = mini.media.get_input_audio_samplerate()
    target_samples = int(record_seconds * turn_in_sr)
    print(f"{record_seconds}秒間、話しかけてください...")

    samples = []
    collected = 0
    while collected < target_samples:
        s = mini.media.get_audio_sample()
        if s is not None:
            samples.append(s)
            collected += len(s)
        else:
            time.sleep(0.01)
    mini.media.stop_recording()

    turn_audio = np.concatenate(samples, axis=0)[:target_samples]
    turn_mono = turn_audio.mean(axis=1).astype(np.float32)

    # 2. VAD で発話区間だけ抽出
    turn_timestamps = get_speech_timestamps(
        torch.from_numpy(turn_mono),
        vad_model,
        sampling_rate=turn_in_sr,
        threshold=0.5,
    )
    if not turn_timestamps:
        print("発話が検出されませんでした。このターンはスキップします。")
        continue

    turn_speech = np.concatenate(
        [turn_mono[seg["start"]:seg["end"]] for seg in turn_timestamps]
    )
    turn_wav_path = f"turn_{turn}.wav"
    sf.write(turn_wav_path, turn_speech, turn_in_sr)

    # 3. STT
    with open(turn_wav_path, "rb") as f:
        turn_transcript = client.audio.transcriptions.create(
            model="gpt-4o-mini-transcribe-2025-12-15",
            file=f,
            language="ja",
        )
    print(f"あなた: {turn_transcript.text}")

    # 4. Response API（previous_response_id で過去の会話を積み上げる、tools でカメラ・動きも呼べる）
    # REACHY_MINI_SYSTEM_PROMPTではなく、play_robot_moveをほぼ毎回使うよう促す
    # REACHY_MINI_EXPRESSIVE_SYSTEM_PROMPT を使う。
    turn_response = client.responses.create(
        model="gpt-5-nano",
        instructions=REACHY_MINI_EXPRESSIVE_SYSTEM_PROMPT,
        input=turn_transcript.text,
        previous_response_id=previous_response_id,
        tools=[CAMERA_TOOL, PLAY_MOVE_TOOL],
    )

    # モデルがツールを呼び出す限り実行し、結果を返し続ける
    turn_function_calls = [
        item for item in turn_response.output if item.type == "function_call"
    ]
    while turn_function_calls:
        turn_outputs = []
        for call in turn_function_calls:
            turn_outputs.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": await run_tool_call(call),
            })

        turn_response = client.responses.create(
            model="gpt-5-nano",
            previous_response_id=turn_response.id,
            input=turn_outputs,
        )
        turn_function_calls = [
            item for item in turn_response.output if item.type == "function_call"
        ]

    previous_response_id = turn_response.id
    print(f"Reachy Mini: {turn_response.output_text}")

    # 5. TTS + 再生
    turn_speech_path = f"turn_{turn}_response.wav"
    with client.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=turn_response.output_text,
        response_format="wav",
    ) as tts_response:
        tts_response.stream_to_file(turn_speech_path)

    mini.media.play_sound(os.path.abspath(turn_speech_path))
    time.sleep(audio_duration_seconds(turn_speech_path) + 0.3)

print("\nデモ終了")